# 10 — Consolidate the verified Hugging Face release

This CPU-only Kaggle notebook creates a **new, non-destructive convenience copy**
of the completed `e2am-memrag-v3r1` experiment. It does not rerun any scientific
experiment and never deletes, renames, rewrites, merges, or force-pushes a source
branch.

## Numbered runbook

1. Create a fresh Kaggle notebook and choose **Accelerator: None**. A GPU provides
   no benefit here.
2. Turn **Internet on**.
3. Add a Kaggle secret named `HF_TOKEN` with write access to
   `Shanmuk4622/E2AM-MemRAG-Traces`. The token is never printed or serialized.
4. Run all cells. Use only one live copy of this consolidation notebook at a time.
5. Expect roughly **8–12 hours**: all 282 artifacts are downloaded from immutable
   commits, checksummed, archived, uploaded, and downloaded again for verification
   under a conservative 90/128 weighted-operations-per-hour ceiling.
6. If Kaggle stops, open a fresh session and run all cells again. The notebook reads
   remote `PROGRESS.json`, verifies completed archives, and resumes at the next branch.
7. A clean or interrupted download creates no unique scientific state. Dirty state
   is committed after every source-branch closure (a major unit); clean heartbeats
   are skipped. `KeyboardInterrupt` verifies the last remote progress seal.
8. Completion requires `CONSOLIDATION_COMPLETE` followed by `FINAL_GO`.

## Output layout

- Original 23 v3r1 stage branches: unchanged and still recoverable.
- Older v2/v3 branches: unchanged, recorded, and excluded from v3r1 evidence.
- New branch: `consolidated-e2am-memrag-v3r1`.
- Human-readable paper data on `main`:
  `experiments/e2am-memrag-v3r1/paper/`.
- Frozen `experiments/e2am-memrag-v3r1/RELEASE.json`: never overwritten.

Experiment completion and hypothesis success are separate. The completed release
passed its closure/fresh-restore gate; the predeclared confirmatory hypothesis did
not pass and remains reported as such.


In [ ]:
# Environment must be frozen before importing Hub/CUDA-aware packages.
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HOME"] = "/kaggle/working/e2am_consolidation/hf-home"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORK_ROOT = Path("/kaggle/working/e2am_consolidation")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("ENVIRONMENT_READY", {"accelerator": "CPU", "work_root": str(WORK_ROOT)})


In [ ]:
# Import first; install only when Kaggle genuinely lacks the required Hub API.
import importlib
import subprocess
import sys
import time

def hub_api_ready():
    try:
        module = importlib.import_module("huggingface_hub")
        return all(hasattr(module, name) for name in (
            "CommitOperationAdd", "HfApi", "hf_hub_download"
        ))
    except Exception:
        return False

if not hub_api_ready():
    last_output = ""
    for attempt, wait_seconds in enumerate((0, 5, 15, 30), start=1):
        if wait_seconds:
            time.sleep(wait_seconds)
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-cache-dir", "--retries", "1", "--timeout", "30", "-q",
             "huggingface-hub>=0.28,<1.0"],
            text=True, capture_output=True,
        )
        last_output = (result.stdout + "\n" + result.stderr)[-1200:]
        if result.returncode == 0:
            importlib.invalidate_caches()
            if hub_api_ready():
                break
        print("DEPENDENCY_RETRY", {"attempt": attempt, "wait_next": wait_seconds})
    else:
        raise RuntimeError(
            "DEPENDENCY_SAFE_STOP: huggingface_hub is unavailable. No branch was "
            "created or changed. Confirm Kaggle Internet is ON and restart Run All. "
            "pip tail: " + last_output
        )

import huggingface_hub
print("DEPENDENCY_READY", {"huggingface_hub": huggingface_hub.__version__})


In [ ]:
# Read the credential only from the Kaggle secret/environment.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception as error:
        raise RuntimeError(
            "CREDENTIAL_SAFE_STOP: add a Kaggle secret named HF_TOKEN with write "
            "access, enable it for this notebook, then restart Run All."
        ) from error
if not HF_TOKEN:
    raise RuntimeError("CREDENTIAL_SAFE_STOP: HF_TOKEN is empty")
os.environ["HF_TOKEN"] = HF_TOKEN
print("CREDENTIAL_READY", {"secret_name": "HF_TOKEN", "value_printed": False})


In [ ]:
# Exact live release lock audited on 2026-07-15. Never replace with branch heads.
SOURCE_RELEASE_LOCK = [{'stage_id': '00',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-00-stage-00-coordinator',
  'commit_sha': '70e81d9c4af5bdb2d2d82df331fafc9ad7fb096f',
  'manifest_sha256': '36471ac0f67dc39060c26b8b8c7d6a5ba49dcc4be6a679ac8a0a383a659878c0',
  'artifact_records': 31,
  'artifact_bytes': 2409907},
 {'stage_id': '00',
  'owner': 'test-vault',
  'branch': 'stage-e2am-memrag-v3r1-00-stage-00-test-vault',
  'commit_sha': 'd36c394d8c904360fa2884dfeeea42d407d09095',
  'manifest_sha256': '2d18da877505fa40fe71836ce5af4fe8a4eed3bf31a7adb1a1b7b1e066efb3ff',
  'artifact_records': 1,
  'artifact_bytes': 23045},
 {'stage_id': '01',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-01-stage-01-coordinator',
  'commit_sha': '6c8d55f9490e6297a27ba77d15720f4f9ab6f4c9',
  'manifest_sha256': '120a8365d3deaa45e02ff011cdf5be83f6f4f9537aebe7c79c366460e0e70224',
  'artifact_records': 36,
  'artifact_bytes': 20736229},
 {'stage_id': '02',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-02-stage-02-coordinator',
  'commit_sha': '82af6953b5813859cb819bc5dd0a8979e0ba43c4',
  'manifest_sha256': 'e9b04ec2074ed4eeb13a7656d6bc8ea2cca790571af97751f04c78ab0e4e839b',
  'artifact_records': 9,
  'artifact_bytes': 9998217},
 {'stage_id': '03',
  'owner': 'lane-00',
  'branch': 'stage-e2am-memrag-v3r1-03-stage-03-lane-00',
  'commit_sha': '67e6eee33de1be767325761348953e6d2c38cf5b',
  'manifest_sha256': 'fce35a9e91d12c2e9a2b0ab059b72e9eebab155195a864a2e1dfd0b5b4e375aa',
  'artifact_records': 9,
  'artifact_bytes': 1892377},
 {'stage_id': '03',
  'owner': 'lane-01',
  'branch': 'stage-e2am-memrag-v3r1-03-stage-03-lane-01',
  'commit_sha': 'f06c718ecb8b0ff25bcf676905fdfa6427cb543a',
  'manifest_sha256': '408bdcf4a933e2b4ce5ede6a32d73b267f13f7f0a3588a7c25e6e03842c3dcc5',
  'artifact_records': 7,
  'artifact_bytes': 962053},
 {'stage_id': '03',
  'owner': 'lane-02',
  'branch': 'stage-e2am-memrag-v3r1-03-stage-03-lane-02',
  'commit_sha': 'f602fc91515a4b078d77d3a4f2adf0a8f149bea0',
  'manifest_sha256': 'c8ce2401d1336be40dba3c8c7229331abe23e7c54bf5493b2d466e0c65d05b8c',
  'artifact_records': 8,
  'artifact_bytes': 1609515},
 {'stage_id': '03',
  'owner': 'lane-03',
  'branch': 'stage-e2am-memrag-v3r1-03-stage-03-lane-03',
  'commit_sha': 'f51fd1407e05df2434caa686cffeb239efad885e',
  'manifest_sha256': '960011716fde4064b4f77f3b818652a1d69534b81e1c7d8ff9c17cd26f1a8cce',
  'artifact_records': 8,
  'artifact_bytes': 1459303},
 {'stage_id': '04',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-04-stage-04-coordinator',
  'commit_sha': '9835b0546ace579006cb78520e610d46fc9df2cb',
  'manifest_sha256': 'ec2069c6c10f9c68fe47c649c3320c5617c1c3a4c23fd68079d073f4315e4a3a',
  'artifact_records': 7,
  'artifact_bytes': 15014694},
 {'stage_id': '05',
  'owner': 'lane-00',
  'branch': 'stage-e2am-memrag-v3r1-05-stage-05-lane-00',
  'commit_sha': 'da21c0deb26113b5b541add8c81845d5780c1f90',
  'manifest_sha256': 'd390c7061c14eab428951fcc1f091e0d8837be2cab4637054ab1ab518015fec4',
  'artifact_records': 28,
  'artifact_bytes': 11600094},
 {'stage_id': '05',
  'owner': 'lane-01',
  'branch': 'stage-e2am-memrag-v3r1-05-stage-05-lane-01',
  'commit_sha': 'd5b4de4408ae904d4c3e316f6f8c69ede6d26a5f',
  'manifest_sha256': 'f337fdac3a9d6287787afc21f3df485f85d713e8752231bc3d974f1df515b7de',
  'artifact_records': 9,
  'artifact_bytes': 1397737},
 {'stage_id': '05',
  'owner': 'lane-02',
  'branch': 'stage-e2am-memrag-v3r1-05-stage-05-lane-02',
  'commit_sha': '5b505707b7baafcd9bd175b8872c28eb4ea4e388',
  'manifest_sha256': 'c19405c66ca3f5b99eb043674a848531bbc25953a57c48bd8d769a78bc139af7',
  'artifact_records': 19,
  'artifact_bytes': 7093174},
 {'stage_id': '05',
  'owner': 'lane-03',
  'branch': 'stage-e2am-memrag-v3r1-05-stage-05-lane-03',
  'commit_sha': '0ba3b1d148f5cc05941f0de6838ba3638ffe3be1',
  'manifest_sha256': '1f2c043ec512dec5d836fbea90c5af40fe444bbbcf52e909e2a249f946d1732a',
  'artifact_records': 14,
  'artifact_bytes': 5895457},
 {'stage_id': '06',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-06-stage-06-coordinator',
  'commit_sha': '1b765103f579e972f90a1fb52f2201cb08c19066',
  'manifest_sha256': '05b72dda0b43de75df0fc580ff764b3f35cd8ad905afbce949d7c6a9dac3304d',
  'artifact_records': 16,
  'artifact_bytes': 20715468},
 {'stage_id': '07',
  'owner': 'lane-00',
  'branch': 'stage-e2am-memrag-v3r1-07-stage-07-lane-00',
  'commit_sha': 'fcceceed653ec4634994acb1d5e14d95dc417b81',
  'manifest_sha256': 'e9cdf3bfe5efae3fbf35008eccb1a6d5a62c9b7a29dece8eb3ceea24cc3e3b65',
  'artifact_records': 12,
  'artifact_bytes': 3706031},
 {'stage_id': '07',
  'owner': 'lane-01',
  'branch': 'stage-e2am-memrag-v3r1-07-stage-07-lane-01',
  'commit_sha': '6c807dfdf5d01c0933a97dd6d0679df886299712',
  'manifest_sha256': '7c277d8381366f9c450259975e0978d2ddf1d3bb7ee7a765b8d7c43123b974a7',
  'artifact_records': 6,
  'artifact_bytes': 281087},
 {'stage_id': '07',
  'owner': 'lane-02',
  'branch': 'stage-e2am-memrag-v3r1-07-stage-07-lane-02',
  'commit_sha': '30a0e834d074502a8b44610385ef24a0be507a5e',
  'manifest_sha256': '51a18c1873141e44a7f7bea8a40076f850b37bdaf719effb647a1d085a896e53',
  'artifact_records': 10,
  'artifact_bytes': 2284158},
 {'stage_id': '07',
  'owner': 'lane-03',
  'branch': 'stage-e2am-memrag-v3r1-07-stage-07-lane-03',
  'commit_sha': '0ffdd4cfbb1e988a61fc0950f4bf36431e67e647',
  'manifest_sha256': 'ce283a0b03e94002b64efc05d67873e8d70b5c285834c69bdfc3cfc23c092afe',
  'artifact_records': 9,
  'artifact_bytes': 2043878},
 {'stage_id': '08',
  'owner': 'lane-00',
  'branch': 'stage-e2am-memrag-v3r1-08-stage-08-lane-00',
  'commit_sha': 'fba12a51f33cb5563349cc9688952d791eb59f1d',
  'manifest_sha256': 'be7e24f11640f595728a3e5a59b4a81a68e8d30d78c9de3133c5fb8e5e87d191',
  'artifact_records': 8,
  'artifact_bytes': 1722275},
 {'stage_id': '08',
  'owner': 'lane-01',
  'branch': 'stage-e2am-memrag-v3r1-08-stage-08-lane-01',
  'commit_sha': '83cabdaeb420472a750ace773c0c1f215c4eca93',
  'manifest_sha256': '355d4181acca88852a61e44253c822abe28c6d3fdc1be0535fcdcea24c0b5c44',
  'artifact_records': 8,
  'artifact_bytes': 1729455},
 {'stage_id': '08',
  'owner': 'lane-02',
  'branch': 'stage-e2am-memrag-v3r1-08-stage-08-lane-02',
  'commit_sha': '25354f0252b38d111b9d05100b091487e4ccb0e7',
  'manifest_sha256': 'c4f0b750a1851c4fddc4cbca865960b5d3416c2c4bc630b15f5b1176bb59d759',
  'artifact_records': 8,
  'artifact_bytes': 1722349},
 {'stage_id': '08',
  'owner': 'lane-03',
  'branch': 'stage-e2am-memrag-v3r1-08-stage-08-lane-03',
  'commit_sha': 'c66f55fa123c85703204ca16cbb8712a792c1b3e',
  'manifest_sha256': '165d4386765030334eb202e0bbcf288f029601558ca3f86636f4b3a30639b0b9',
  'artifact_records': 8,
  'artifact_bytes': 1729828},
 {'stage_id': '09',
  'owner': 'coordinator',
  'branch': 'stage-e2am-memrag-v3r1-09-stage-09-coordinator',
  'commit_sha': '4d3e111ddfe41247b511ad0fc1a413baacee7864',
  'manifest_sha256': 'c67191cf2afcc1a9d2be530d76049621d67f8b45528fc341bbf8f936cb6ecdf3',
  'artifact_records': 11,
  'artifact_bytes': 11528142}]

RELEASE_POINTER_LOCK = {
    "experiment_id": "e2am-memrag-v3r1",
    "artifact_prefix": "experiments/e2am-memrag-v3r1/stages/09/coordinator",
    "stage_branch": "stage-e2am-memrag-v3r1-09-stage-09-coordinator",
    "stage_commit_sha": "4d3e111ddfe41247b511ad0fc1a413baacee7864",
    "release_manifest_sha256": "b1627b04695f502aec75d1424b57a42f0928446cd35c6d3302d69f8a58ab685f",
    "success_gate": "_SUCCESS.json",
    "success_gate_sha256": "f85ca74fa96d3eb327772a43b596637ed8810f1c04029e49d2623a1e1f7a12ef",
}

CONFIG = {
    "repo_id": "Shanmuk4622/E2AM-MemRAG-Traces",
    "experiment_id": "e2am-memrag-v3r1",
    "destination_branch": "consolidated-e2am-memrag-v3r1",
    "remote_root": "consolidated/e2am-memrag-v3r1",
    "work_root": str(WORK_ROOT),
    "hub_capacity": 90,
    "dirty_sync_target_seconds": 1200,
    "expected_artifact_records": 282,
    "expected_artifact_bytes": 127_554_473,
    "source_release_lock": SOURCE_RELEASE_LOCK,
    "release_pointer_lock": RELEASE_POINTER_LOCK,
}
print("RELEASE_LOCK_READY", {
    "branches": len(SOURCE_RELEASE_LOCK),
    "artifact_records": sum(x["artifact_records"] for x in SOURCE_RELEASE_LOCK),
    "artifact_bytes": sum(x["artifact_bytes"] for x in SOURCE_RELEASE_LOCK),
})


In [ ]:
"""Standalone, non-destructive Hugging Face consolidation runtime.

This file is embedded verbatim into the Stage-10 Kaggle notebook.  It deliberately
lives outside ``src/e2am_memrag`` so publishing a post-experiment convenience copy
cannot change the frozen scientific source-tree hash.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import time
import zipfile
from collections import deque
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any, Callable, Iterable, Mapping


SHA256_RE = re.compile(r"^[0-9a-f]{64}$")
COMMIT_RE = re.compile(r"^[0-9a-f]{40}$")
TOKEN_RE = re.compile(rb"hf_[A-Za-z0-9]{20,}")
MANAGED_README_MARKER = "<!-- E2AM-MEMRAG-CONSOLIDATOR -->"


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace(
        "+00:00", "Z"
    )


def canonical_json(value: Any) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def safe_relative_path(value: str) -> PurePosixPath:
    if not isinstance(value, str) or not value or "\\" in value:
        raise ValueError(f"Unsafe logical path: {value!r}")
    path = PurePosixPath(value)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in path.parts):
        raise ValueError(f"Unsafe logical path: {value!r}")
    return path


def safe_target(root: Path, logical_path: str) -> Path:
    relative = safe_relative_path(logical_path)
    target = root.joinpath(*relative.parts)
    resolved_root = root.resolve()
    resolved_target = target.resolve()
    if resolved_target != resolved_root and resolved_root not in resolved_target.parents:
        raise ValueError(f"Artifact escapes restore root: {logical_path!r}")
    return target


def atomic_write(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    try:
        with temporary.open("xb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def deterministic_zip(source_root: Path, destination: Path) -> dict[str, Any]:
    files = sorted(path for path in source_root.rglob("*") if path.is_file())
    if not files:
        raise RuntimeError(f"Refusing to archive an empty closure: {source_root}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        f".{destination.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )
    inventory: list[dict[str, Any]] = []
    try:
        with zipfile.ZipFile(
            temporary,
            mode="x",
            compression=zipfile.ZIP_DEFLATED,
            compresslevel=6,
            allowZip64=True,
        ) as archive:
            for path in files:
                logical = path.relative_to(source_root).as_posix()
                payload = path.read_bytes()
                info = zipfile.ZipInfo(logical, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                info.create_system = 3
                info.external_attr = 0o100644 << 16
                archive.writestr(
                    info,
                    payload,
                    compress_type=zipfile.ZIP_DEFLATED,
                    compresslevel=6,
                )
                inventory.append(
                    {
                        "logical_path": logical,
                        "sha256": sha256_bytes(payload),
                        "bytes": len(payload),
                    }
                )
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)
    return {
        "path": str(destination),
        "sha256": sha256_file(destination),
        "bytes": destination.stat().st_size,
        "files": inventory,
    }


def extract_verified_zip(archive_path: Path, destination: Path) -> None:
    temporary = destination.with_name(
        f".{destination.name}.extract-{os.getpid()}-{time.time_ns()}"
    )
    if temporary.exists():
        shutil.rmtree(temporary)
    temporary.mkdir(parents=True)
    try:
        with zipfile.ZipFile(archive_path, mode="r") as archive:
            names = [info.filename for info in archive.infolist()]
            if len(names) != len(set(names)):
                raise RuntimeError("Archive contains duplicate members")
            for info in archive.infolist():
                logical = safe_relative_path(info.filename)
                if info.is_dir():
                    raise RuntimeError("Consolidated branch archives contain files only")
                target = temporary.joinpath(*logical.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                atomic_write(target, archive.read(info))
        if destination.exists():
            shutil.rmtree(destination)
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)


class RollingHubBudget:
    """A visible rolling limiter with 90/128 weighted operations per hour."""

    def __init__(
        self,
        *,
        capacity: int = 90,
        window_seconds: float = 3600.0,
        clock: Callable[[], float] = time.time,
        sleeper: Callable[[float], None] = time.sleep,
    ) -> None:
        if capacity > 96 or capacity < 1:
            raise ValueError("Hub capacity must preserve at least 25% of 128 requests/hour")
        self.capacity = int(capacity)
        self.window_seconds = float(window_seconds)
        self.clock = clock
        self.sleeper = sleeper
        self.events: deque[tuple[float, int]] = deque()

    def restore(self, events: Iterable[Iterable[Any]]) -> None:
        now = self.clock()
        restored: list[tuple[float, int]] = []
        for raw in events:
            values = list(raw)
            if len(values) != 2:
                continue
            timestamp, weight = float(values[0]), int(values[1])
            if 0 < weight <= self.capacity and now - timestamp < self.window_seconds:
                restored.append((timestamp, weight))
        self.events = deque(sorted(restored))
        self._prune()

    def snapshot(self) -> list[list[float | int]]:
        self._prune()
        return [[timestamp, weight] for timestamp, weight in self.events]

    def _prune(self) -> None:
        threshold = self.clock() - self.window_seconds
        while self.events and self.events[0][0] <= threshold:
            self.events.popleft()

    @property
    def used(self) -> int:
        self._prune()
        return sum(weight for _, weight in self.events)

    def acquire(self, weight: int, *, reason: str) -> None:
        if not 0 < weight <= self.capacity:
            raise ValueError("Invalid Hub budget weight")
        last_notice = 0.0
        while True:
            self._prune()
            if self.used + weight <= self.capacity:
                self.events.append((self.clock(), weight))
                return
            oldest = self.events[0][0]
            wait_seconds = max(1.0, oldest + self.window_seconds - self.clock() + 1.0)
            if self.clock() - last_notice >= 60.0:
                print(
                    "HUB_BUDGET_WAIT",
                    {
                        "reason": reason,
                        "used_weight": self.used,
                        "capacity": self.capacity,
                        "remaining_seconds": round(wait_seconds, 1),
                    },
                    flush=True,
                )
                last_notice = self.clock()
            self.sleeper(min(30.0, wait_seconds))


def http_status(error: BaseException) -> int | None:
    response = getattr(error, "response", None)
    status = getattr(response, "status_code", None)
    return int(status) if isinstance(status, int) else None


def retry_after(error: BaseException) -> float | None:
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", None)
    value = headers.get("Retry-After") if hasattr(headers, "get") else None
    try:
        return max(0.0, float(value)) if value is not None else None
    except (TypeError, ValueError):
        return None


class Consolidator:
    def __init__(self, config: Mapping[str, Any], *, hf_token: str) -> None:
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("HF_TOKEN is required for publishing")
        self.config = dict(config)
        self.hf_token = hf_token.strip()
        self.repo_id = str(self.config["repo_id"])
        self.repo_type = "dataset"
        self.experiment_id = str(self.config["experiment_id"])
        self.destination_branch = str(self.config["destination_branch"])
        self.remote_root = str(self.config["remote_root"])
        self.dirty_sync_target_seconds = int(
            self.config.get("dirty_sync_target_seconds", 1200)
        )
        if self.dirty_sync_target_seconds != 1200:
            raise RuntimeError("Dirty-state sync target must remain frozen at 1,200 seconds")
        self.work_root = Path(self.config["work_root"])
        self.cache_root = self.work_root / "hub-cache"
        self.source_root = self.work_root / "source"
        self.bundle_root = self.work_root / "bundles"
        self.paper_root = self.work_root / "paper"
        self.state_root = self.work_root / "state"
        for root in (
            self.work_root,
            self.cache_root,
            self.source_root,
            self.bundle_root,
            self.paper_root,
            self.state_root,
        ):
            root.mkdir(parents=True, exist_ok=True)

        self.source_lock = [dict(item) for item in self.config["source_release_lock"]]
        self.source_lock_sha256 = sha256_bytes(canonical_json(self.source_lock))
        self.expected_branches = {item["branch"]: item for item in self.source_lock}
        if len(self.expected_branches) != len(self.source_lock):
            raise RuntimeError("Source release lock repeats a branch")
        self._validate_source_lock()
        self.budget = RollingHubBudget(capacity=int(self.config.get("hub_capacity", 90)))
        self.api = None
        self.hf_hub_download = None
        self.CommitOperationAdd = None
        self.progress: dict[str, Any] | None = None
        self.destination_head: str | None = None
        self.last_verified_progress_bytes: bytes | None = None

    def _validate_source_lock(self) -> None:
        total_records = 0
        total_bytes = 0
        for item in self.source_lock:
            required = {
                "stage_id",
                "owner",
                "branch",
                "commit_sha",
                "manifest_sha256",
                "artifact_records",
                "artifact_bytes",
            }
            if set(item) != required:
                raise RuntimeError("Source release lock schema is invalid")
            stage = str(item["stage_id"])
            owner = str(item["owner"])
            expected_branch = (
                f"stage-{self.experiment_id}-{stage}-stage-{stage}-{owner}"
            )
            if item["branch"] != expected_branch:
                raise RuntimeError(f"Source branch identity is invalid: {item['branch']}")
            if not COMMIT_RE.fullmatch(str(item["commit_sha"])):
                raise RuntimeError("Source release lock has a non-immutable commit")
            if not SHA256_RE.fullmatch(str(item["manifest_sha256"])):
                raise RuntimeError("Source release lock has an invalid manifest digest")
            total_records += int(item["artifact_records"])
            total_bytes += int(item["artifact_bytes"])
        if total_records != int(self.config["expected_artifact_records"]):
            raise RuntimeError("Frozen artifact-record total is inconsistent")
        if total_bytes != int(self.config["expected_artifact_bytes"]):
            raise RuntimeError("Frozen artifact-byte total is inconsistent")

    def _load_hub(self) -> None:
        from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download

        self.api = HfApi(token=self.hf_token)
        self.hf_hub_download = hf_hub_download
        self.CommitOperationAdd = CommitOperationAdd

    def _call(
        self,
        operation: Callable[[], Any],
        *,
        weight: int,
        reason: str,
        public_download: bool = False,
    ) -> Any:
        for attempt in range(5):
            # Each retry is a real Hub operation and therefore consumes budget.
            self.budget.acquire(weight, reason=reason)
            try:
                return operation()
            except Exception as error:
                status = http_status(error)
                # Public Hub downloads can occasionally receive an expired or invalid
                # signed object-store URL.  Preserve that raw 403 so _download can
                # invalidate the cached URL and retry it.  Authenticated API failures
                # remain immediate, non-retryable safe stops.
                if status == 403 and public_download:
                    raise
                if status in {401, 403}:
                    raise RuntimeError(
                        "HUB_AUTHENTICATION_STOP: correct the Kaggle HF_TOKEN; no retry "
                        "or source-branch mutation was attempted"
                    ) from error
                retryable = status == 429 or (
                    status is not None and 500 <= status < 600
                )
                if not retryable or attempt == 4:
                    raise
                delay = retry_after(error)
                if delay is None:
                    delay = min(60.0, 2.0**attempt)
                print(
                    "HUB_TRANSIENT_RETRY",
                    {"reason": reason, "attempt": attempt + 1, "wait_seconds": delay},
                    flush=True,
                )
                time.sleep(delay)
        raise RuntimeError("Unreachable retry state")

    def _download(
        self,
        *,
        filename: str,
        revision: str,
        public_read: bool = True,
    ) -> bytes:
        safe_relative_path(filename)
        token: str | bool = False if public_read else self.hf_token

        def perform(force_download: bool = False) -> str:
            return self.hf_hub_download(
                repo_id=self.repo_id,
                repo_type=self.repo_type,
                revision=revision,
                filename=filename,
                token=token,
                cache_dir=str(self.cache_root),
                force_download=force_download,
            )

        try:
            local = self._call(
                perform,
                weight=2,
                reason=f"download:{filename}",
                public_download=public_read,
            )
        except Exception as error:
            text = str(error).lower()
            signed_url_failure = http_status(error) == 403 and any(
                marker in text
                for marker in ("signature", "expired", "xet", "accessdenied")
            )
            if not signed_url_failure or not public_read:
                raise
            print(
                "PUBLIC_SIGNED_URL_REFRESH",
                {"filename": filename, "revision": revision},
                flush=True,
            )
            local = self._call(
                lambda: perform(True),
                weight=2,
                reason=f"signed-url-refresh:{filename}",
                public_download=True,
            )
        return Path(local).read_bytes()

    def _try_download(
        self, *, filename: str, revision: str, public_read: bool = True
    ) -> bytes | None:
        try:
            return self._download(
                filename=filename,
                revision=revision,
                public_read=public_read,
            )
        except Exception as error:
            if http_status(error) == 404 or type(error).__name__ == "EntryNotFoundError":
                return None
            raise

    def _repo_head(self, revision: str) -> str:
        info = self._call(
            lambda: self.api.repo_info(
                repo_id=self.repo_id,
                repo_type=self.repo_type,
                revision=revision,
            ),
            weight=1,
            reason=f"repo-head:{revision}",
        )
        commit = str(info.sha).strip().lower()
        if not COMMIT_RE.fullmatch(commit):
            raise RuntimeError(f"Hub revision is not immutable: {revision}")
        return commit

    def _list_refs(self) -> tuple[dict[str, str], list[str]]:
        refs = self._call(
            lambda: self.api.list_repo_refs(
                repo_id=self.repo_id,
                repo_type=self.repo_type,
            ),
            weight=1,
            reason="list-repository-refs",
        )
        branches: dict[str, str] = {}
        for branch in refs.branches:
            name = str(branch.name)
            commit = str(branch.target_commit).strip().lower()
            if not COMMIT_RE.fullmatch(commit):
                raise RuntimeError(f"Non-immutable branch head: {name}")
            branches[name] = commit
        missing = sorted(set(self.expected_branches) - set(branches))
        if missing:
            raise RuntimeError("SOURCE_BRANCHES_MISSING: " + ", ".join(missing))
        legacy = sorted(
            name
            for name in branches
            if name.startswith("stage-e2am-memrag-")
            and name not in self.expected_branches
        )
        advanced = {
            name: {"locked": self.expected_branches[name]["commit_sha"], "head": branches[name]}
            for name in self.expected_branches
            if branches[name] != self.expected_branches[name]["commit_sha"]
        }
        if advanced:
            print("SOURCE_HEADS_ADVANCED_PINNED_COMMITS_PRESERVED", advanced, flush=True)
        print(
            "SOURCE_BRANCH_AUDIT",
            {
                "required_v3r1": len(self.expected_branches),
                "excluded_legacy": legacy,
                "advanced_but_pinned": len(advanced),
            },
            flush=True,
        )
        return branches, legacy

    def _verify_release_pointer(self) -> dict[str, Any]:
        release_path = f"experiments/{self.experiment_id}/RELEASE.json"
        payload = self._download(filename=release_path, revision="main")
        release = json.loads(payload)
        expected = dict(self.config["release_pointer_lock"])
        for key, value in expected.items():
            if release.get(key) != value:
                raise RuntimeError(f"Main RELEASE.json disagrees on {key}")
        print(
            "MAIN_RELEASE_POINTER_VERIFIED",
            {
                "stage_commit_sha": release["stage_commit_sha"],
                "success_gate_sha256": release["success_gate_sha256"],
            },
            flush=True,
        )
        return release

    def _new_progress(self, *, legacy_branches: list[str]) -> dict[str, Any]:
        now = utc_now()
        return {
            "schema_version": 1,
            "experiment_id": self.experiment_id,
            "source_lock_sha256": self.source_lock_sha256,
            "destination_branch": self.destination_branch,
            "dirty_sync_target_seconds": self.dirty_sync_target_seconds,
            "status": "RUNNING",
            "started_at": now,
            "updated_at": now,
            "completed": {},
            "excluded_legacy_branches": legacy_branches,
            # Reserve the initialization commit and its remote verification so an
            # immediate fresh-session resume cannot undercount recent Hub traffic.
            "hub_budget_events": self._budget_snapshot_with_reservations(16),
        }

    def _budget_snapshot_with_reservations(self, *weights: int) -> list[list[float | int]]:
        events = self.budget.snapshot()
        timestamp = time.time()
        for weight in weights:
            if not 0 < int(weight) <= self.budget.capacity:
                raise ValueError("Invalid persisted Hub-budget reservation")
            events.append([timestamp, int(weight)])
        return events

    def _validate_progress(self, progress: Mapping[str, Any]) -> dict[str, Any]:
        value = dict(progress)
        if (
            value.get("schema_version") != 1
            or value.get("experiment_id") != self.experiment_id
            or value.get("source_lock_sha256") != self.source_lock_sha256
            or value.get("destination_branch") != self.destination_branch
            or value.get("dirty_sync_target_seconds") != self.dirty_sync_target_seconds
            or value.get("status") not in {"RUNNING", "COMPLETE"}
            or not isinstance(value.get("completed"), dict)
        ):
            raise RuntimeError("Destination PROGRESS.json belongs to another consolidation")
        unknown = set(value["completed"]) - set(self.expected_branches)
        if unknown:
            raise RuntimeError("Destination progress contains unknown source branches")
        for branch, entry in value["completed"].items():
            lock = self.expected_branches[branch]
            if (
                entry.get("source_commit_sha") != lock["commit_sha"]
                or entry.get("source_manifest_sha256") != lock["manifest_sha256"]
                or entry.get("artifact_records") != lock["artifact_records"]
                or entry.get("artifact_bytes") != lock["artifact_bytes"]
                or not SHA256_RE.fullmatch(str(entry.get("archive_sha256", "")))
                or not isinstance(entry.get("artifact_inventory"), list)
            ):
                raise RuntimeError(f"Completed branch receipt is invalid: {branch}")
        return value

    def _local_file(self, name: str, payload: bytes) -> Path:
        path = self.state_root / name
        if TOKEN_RE.search(payload) or self.hf_token.encode("utf-8") in payload:
            raise RuntimeError("Secret-like value rejected from publication metadata")
        atomic_write(path, payload)
        return path

    def _commit_verified(
        self,
        *,
        revision: str,
        parent_commit: str,
        files: Mapping[str, Path],
        message: str,
        allow_existing_equal: bool = True,
    ) -> str:
        if not files:
            return parent_commit
        for remote_path, local_path in files.items():
            safe_relative_path(remote_path)
            if not local_path.is_file() or local_path.is_symlink():
                raise RuntimeError(f"Publication source is not a regular file: {local_path}")
        current_head = self._repo_head(revision)
        if allow_existing_equal:
            all_equal = True
            for remote_path, local_path in files.items():
                remote = self._try_download(
                    filename=remote_path,
                    revision=current_head,
                    public_read=True,
                )
                if remote != local_path.read_bytes():
                    all_equal = False
                    break
            if all_equal:
                return current_head
        if current_head != parent_commit:
            all_equal = True
            for remote_path, local_path in files.items():
                remote = self._try_download(
                    filename=remote_path,
                    revision=current_head,
                    public_read=True,
                )
                all_equal = all_equal and remote == local_path.read_bytes()
            if allow_existing_equal and all_equal:
                return current_head
            raise RuntimeError(
                f"SECOND_WRITER_STOP: {revision} advanced from {parent_commit} to {current_head}"
            )

        operations = [
            self.CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
            for remote, local in sorted(files.items())
        ]
        try:
            commit = self._call(
                lambda: self.api.create_commit(
                    repo_id=self.repo_id,
                    repo_type=self.repo_type,
                    revision=revision,
                    parent_commit=parent_commit,
                    operations=operations,
                    commit_message=message,
                ),
                weight=4,
                reason=f"create-commit:{revision}",
            )
            commit_sha = str(
                getattr(commit, "oid", None) or getattr(commit, "commit_id", None)
            ).strip().lower()
        except Exception as error:
            if http_status(error) != 409:
                raise
            commit_sha = self._repo_head(revision)
        if not COMMIT_RE.fullmatch(commit_sha):
            commit_sha = self._repo_head(revision)

        for remote_path, local_path in files.items():
            remote = self._download(
                filename=remote_path,
                revision=commit_sha,
                public_read=True,
            )
            if remote != local_path.read_bytes():
                raise RuntimeError(f"REMOTE_UPLOAD_VERIFICATION_FAILED: {remote_path}")
        return commit_sha

    def _initialize_destination(self, *, branches: Mapping[str, str], legacy: list[str]) -> None:
        main_head = branches["main"]
        created = self.destination_branch not in branches
        if created:
            self._call(
                lambda: self.api.create_branch(
                    repo_id=self.repo_id,
                    repo_type=self.repo_type,
                    branch=self.destination_branch,
                    revision=main_head,
                    exist_ok=True,
                ),
                weight=2,
                reason="create-consolidation-branch",
            )
        self.destination_head = self._repo_head(self.destination_branch)
        progress_path = f"{self.remote_root}/PROGRESS.json"
        raw = self._try_download(
            filename=progress_path,
            revision=self.destination_head,
            public_read=True,
        )
        if raw is None:
            repo_files = self._call(
                lambda: self.api.list_repo_files(
                    repo_id=self.repo_id,
                    repo_type=self.repo_type,
                    revision=self.destination_head,
                ),
                weight=1,
                reason="inspect-consolidation-branch",
            )
            managed = [path for path in repo_files if path.startswith(self.remote_root + "/")]
            if managed:
                raise RuntimeError(
                    "Destination branch has managed files but no PROGRESS.json; refusing overwrite"
                )
            self.progress = self._new_progress(legacy_branches=legacy)
            progress_bytes = canonical_json(self.progress)
            progress_local = self._local_file("PROGRESS.json", progress_bytes)
            readme_local = self._local_file(
                "BRANCH_README.md", self._branch_readme(status="RUNNING").encode("utf-8")
            )
            self.destination_head = self._commit_verified(
                revision=self.destination_branch,
                parent_commit=self.destination_head,
                files={
                    progress_path: progress_local,
                    f"{self.remote_root}/README.md": readme_local,
                },
                message=f"Initialize {self.experiment_id} non-destructive consolidation",
            )
            self.last_verified_progress_bytes = progress_bytes
        else:
            self.progress = self._validate_progress(json.loads(raw))
            self.budget.restore(self.progress.get("hub_budget_events", []))
            self.last_verified_progress_bytes = raw
            print(
                "CONSOLIDATION_RESUME",
                {
                    "status": self.progress["status"],
                    "completed_branches": len(self.progress["completed"]),
                    "destination_commit": self.destination_head,
                },
                flush=True,
            )

    def _validate_pointer_and_manifest(
        self, lock: Mapping[str, Any]
    ) -> tuple[dict[str, Any], list[dict[str, Any]]]:
        stage = str(lock["stage_id"])
        owner = str(lock["owner"])
        commit = str(lock["commit_sha"])
        prefix = f"experiments/{self.experiment_id}/stages/{stage}/{owner}"
        pointer_path = f"{prefix}/LATEST.json"
        pointer_raw = self._download(filename=pointer_path, revision=commit)
        pointer = json.loads(pointer_raw)
        binding = {
            "repo_id": self.repo_id,
            "repo_type": self.repo_type,
            "experiment_id": self.experiment_id,
            "worker_id": f"stage-{stage}-{owner}",
            "branch": lock["branch"],
            "base_revision": "main",
            "remote_prefix": prefix,
        }
        expected_manifest_path = (
            f"{prefix}/manifests/{lock['manifest_sha256']}.json"
        )
        if (
            set(pointer)
            != {"schema_version", "binding", "manifest_path", "manifest_sha256"}
            or pointer.get("schema_version") != 1
            or pointer.get("binding") != binding
            or pointer.get("manifest_path") != expected_manifest_path
            or pointer.get("manifest_sha256") != lock["manifest_sha256"]
        ):
            raise RuntimeError(f"Invalid source pointer: {lock['branch']}")
        manifest_raw = self._download(
            filename=pointer["manifest_path"], revision=commit
        )
        if sha256_bytes(manifest_raw) != lock["manifest_sha256"]:
            raise RuntimeError(f"Source manifest checksum mismatch: {lock['branch']}")
        manifest = json.loads(manifest_raw)
        records = manifest.get("artifacts")
        if (
            manifest.get("schema_version") != 1
            or manifest.get("binding") != binding
            or not isinstance(records, list)
            or len(records) != lock["artifact_records"]
        ):
            raise RuntimeError(f"Source manifest identity mismatch: {lock['branch']}")
        validated: list[dict[str, Any]] = []
        logical_seen: set[str] = set()
        for record in records:
            logical = str(record.get("logical_path", ""))
            digest = str(record.get("sha256", ""))
            byte_count = record.get("bytes")
            safe_relative_path(logical)
            expected_remote = f"{prefix}/artifacts/sha256/{digest[:2]}/{digest}"
            if (
                set(record)
                != {"logical_path", "remote_path", "sha256", "bytes", "metadata"}
                or logical in logical_seen
                or not SHA256_RE.fullmatch(digest)
                or isinstance(byte_count, bool)
                or not isinstance(byte_count, int)
                or byte_count < 0
                or record.get("remote_path") != expected_remote
            ):
                raise RuntimeError(f"Malformed source artifact record: {logical}")
            logical_seen.add(logical)
            canonical_json(record.get("metadata"))
            validated.append(dict(record))
        if validated != sorted(validated, key=lambda item: item["logical_path"]):
            raise RuntimeError("Source manifest artifact order is not canonical")
        if sum(record["bytes"] for record in validated) != lock["artifact_bytes"]:
            raise RuntimeError(f"Source artifact-byte total mismatch: {lock['branch']}")
        return pointer, validated

    def _restore_source(self, lock: Mapping[str, Any]) -> tuple[Path, list[dict[str, Any]]]:
        branch = str(lock["branch"])
        stage = str(lock["stage_id"])
        owner = str(lock["owner"])
        destination = self.source_root / stage / owner
        temporary = destination.with_name(
            f".{destination.name}.restore-{os.getpid()}-{time.time_ns()}"
        )
        if temporary.exists():
            shutil.rmtree(temporary)
        temporary.mkdir(parents=True)
        try:
            _, records = self._validate_pointer_and_manifest(lock)
            for index, record in enumerate(records, start=1):
                payload = self._download(
                    filename=record["remote_path"],
                    revision=lock["commit_sha"],
                )
                if len(payload) != record["bytes"] or sha256_bytes(payload) != record["sha256"]:
                    raise RuntimeError(
                        f"SOURCE_ARTIFACT_CHECKSUM_FAILED: {branch}/{record['logical_path']}"
                    )
                target = safe_target(temporary, record["logical_path"])
                atomic_write(target, payload)
                if index % 10 == 0 or index == len(records):
                    print(
                        "SOURCE_RESTORE_PROGRESS",
                        {"branch": branch, "verified": index, "total": len(records)},
                        flush=True,
                    )
            if destination.exists():
                shutil.rmtree(destination)
            destination.parent.mkdir(parents=True, exist_ok=True)
            os.replace(temporary, destination)
            return destination, records
        finally:
            if temporary.exists():
                shutil.rmtree(temporary)

    def _receipt_bytes(
        self,
        lock: Mapping[str, Any],
        records: list[dict[str, Any]],
        archive: Mapping[str, Any],
    ) -> bytes:
        return canonical_json(
            {
                "schema_version": 1,
                "experiment_id": self.experiment_id,
                "source_branch": lock["branch"],
                "source_commit_sha": lock["commit_sha"],
                "source_manifest_sha256": lock["manifest_sha256"],
                "artifact_records": len(records),
                "artifact_bytes": sum(record["bytes"] for record in records),
                "archive_sha256": archive["sha256"],
                "archive_bytes": archive["bytes"],
                "artifact_inventory": [
                    {
                        "logical_path": record["logical_path"],
                        "sha256": record["sha256"],
                        "bytes": record["bytes"],
                    }
                    for record in records
                ],
                "verified_at": utc_now(),
                "source_branches_modified": False,
            }
        )

    def _verify_completed_entry(self, branch: str, entry: Mapping[str, Any]) -> None:
        archive = self._download(
            filename=entry["archive_path"],
            revision=self.destination_head,
        )
        receipt = self._download(
            filename=entry["receipt_path"],
            revision=self.destination_head,
        )
        if sha256_bytes(archive) != entry["archive_sha256"]:
            raise RuntimeError(f"Completed archive failed resume verification: {branch}")
        if sha256_bytes(receipt) != entry["receipt_sha256"]:
            raise RuntimeError(f"Completed receipt failed resume verification: {branch}")

    def _process_branch(self, lock: Mapping[str, Any]) -> None:
        branch = str(lock["branch"])
        existing = self.progress["completed"].get(branch)
        if existing is not None:
            self._verify_completed_entry(branch, existing)
            print("SOURCE_BRANCH_ALREADY_CONSOLIDATED", {"branch": branch}, flush=True)
            return

        print(
            "SOURCE_BRANCH_CONSOLIDATION_START",
            {
                "branch": branch,
                "commit": lock["commit_sha"],
                "artifact_records": lock["artifact_records"],
                "artifact_mib": round(lock["artifact_bytes"] / (1024**2), 3),
            },
            flush=True,
        )
        restored, records = self._restore_source(lock)
        archive_path = self.bundle_root / f"{branch}-{lock['commit_sha']}.zip"
        archive = deterministic_zip(restored, archive_path)
        if (
            len(archive["files"]) != lock["artifact_records"]
            or sum(item["bytes"] for item in archive["files"]) != lock["artifact_bytes"]
        ):
            raise RuntimeError(f"Deterministic archive coverage mismatch: {branch}")
        receipt_bytes = self._receipt_bytes(lock, records, archive)
        receipt_local = self._local_file(
            f"{branch}.receipt.json", receipt_bytes
        )
        stage = str(lock["stage_id"])
        owner = str(lock["owner"])
        archive_remote = (
            f"{self.remote_root}/source-branches/{stage}/{owner}/"
            f"{lock['commit_sha']}.zip"
        )
        receipt_remote = (
            f"{self.remote_root}/source-branches/{stage}/{owner}/"
            f"{lock['commit_sha']}.receipt.json"
        )
        entry = {
            "stage_id": stage,
            "owner": owner,
            "source_commit_sha": lock["commit_sha"],
            "source_manifest_sha256": lock["manifest_sha256"],
            "artifact_records": lock["artifact_records"],
            "artifact_bytes": lock["artifact_bytes"],
            "artifact_inventory": [
                {
                    "logical_path": record["logical_path"],
                    "sha256": record["sha256"],
                    "bytes": record["bytes"],
                }
                for record in records
            ],
            "archive_path": archive_remote,
            "archive_sha256": archive["sha256"],
            "archive_bytes": archive["bytes"],
            "receipt_path": receipt_remote,
            "receipt_sha256": sha256_bytes(receipt_bytes),
        }
        candidate_progress = json.loads(canonical_json(self.progress))
        candidate_progress["completed"][branch] = entry
        candidate_progress["updated_at"] = utc_now()
        # Covers repo-head, missing-file probe, commit, and three fresh downloads.
        candidate_progress["hub_budget_events"] = (
            self._budget_snapshot_with_reservations(16)
        )
        progress_bytes = canonical_json(candidate_progress)
        progress_local = self._local_file("PROGRESS.json", progress_bytes)
        self.destination_head = self._commit_verified(
            revision=self.destination_branch,
            parent_commit=self.destination_head,
            files={
                archive_remote: archive_path,
                receipt_remote: receipt_local,
                f"{self.remote_root}/PROGRESS.json": progress_local,
            },
            message=(
                f"Consolidate {self.experiment_id} source branch {stage}/{owner}"
            ),
        )
        self.progress = candidate_progress
        self.last_verified_progress_bytes = progress_bytes
        print(
            "SOURCE_BRANCH_CONSOLIDATION_VERIFIED",
            {
                "branch": branch,
                "destination_commit": self.destination_head,
                "archive_sha256": archive["sha256"],
            },
            flush=True,
        )
        # Original immutable branch plus the freshly verified consolidated archive
        # are now two remote copies, so local materialization is no longer the sole copy.
        shutil.rmtree(restored)
        archive_path.unlink(missing_ok=True)

    def _paper_files_from_stage09(self) -> dict[str, Path]:
        stage09_branch = f"stage-{self.experiment_id}-09-stage-09-coordinator"
        entry = self.progress["completed"][stage09_branch]
        archive_bytes = self._download(
            filename=entry["archive_path"], revision=self.destination_head
        )
        if sha256_bytes(archive_bytes) != entry["archive_sha256"]:
            raise RuntimeError("Stage-09 consolidated archive failed verification")
        archive_local = self.bundle_root / "stage09-paper-source.zip"
        atomic_write(archive_local, archive_bytes)
        extracted = self.work_root / "stage09-paper-source"
        extract_verified_zip(archive_local, extracted)
        expected = [
            "HYPOTHESIS_RESULT.json",
            "RELEASE_CANDIDATE.json",
            "_SUCCESS.json",
            "release/clean_traces.jsonl",
            "release/experiment_summary.json",
            "release/mechanism_analysis.json",
            "release/model_transfer_panel.json",
            "release/robustness_analysis.json",
            "release/robustness_traces.jsonl",
            "release/route_cards.json",
            "release_manifest.json",
        ]
        result: dict[str, Path] = {}
        if self.paper_root.exists():
            shutil.rmtree(self.paper_root)
        self.paper_root.mkdir(parents=True)
        inventory = {item["logical_path"]: item for item in entry["artifact_inventory"]}
        for logical in expected:
            source = safe_target(extracted, logical)
            record = inventory.get(logical)
            if (
                record is None
                or not source.is_file()
                or source.stat().st_size != record["bytes"]
                or sha256_file(source) != record["sha256"]
            ):
                raise RuntimeError(f"Paper artifact is missing or corrupt: {logical}")
            target = safe_target(self.paper_root, logical)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
            result[logical] = target
        return result

    def _branch_readme(self, *, status: str) -> str:
        return f"""# E2AM-MemRAG verified consolidated release

{MANAGED_README_MARKER}

Status: **{status}**

This branch is a post-experiment, non-destructive convenience copy for
`{self.experiment_id}`. Each source worker closure is preserved as a deterministic
ZIP plus a checksum receipt. Original stage branches are never deleted, renamed,
rewritten, or force-pushed. Legacy v2/v3 branches are recorded but excluded from
the v3r1 scientific release.

Paper-facing Stage-09 artifacts are available individually under `paper/`.
`_SUCCESS.json` means the frozen experiment and fresh-restore audit completed;
the predeclared hypothesis outcome is stored separately in
`paper/HYPOTHESIS_RESULT.json`.
"""

    def _paper_readme(self, hypothesis_pass: bool) -> str:
        outcome = "passed" if hypothesis_pass else "did not pass"
        return f"""# Paper-facing E2AM-MemRAG results

{MANAGED_README_MARKER}

The complete frozen experiment finished and passed its closure/fresh-restore audit.
The separately predeclared confirmatory hypothesis **{outcome}**. Do not describe
experiment completion as hypothesis success.

Key files:

- `experiment_summary.json`: primary policy and route summaries.
- `mechanism_analysis.json`: retrieval and memory mechanism contrasts.
- `model_transfer_panel.json`: five-model direct/grounded panel and Pareto data.
- `robustness_analysis.json`: clean-versus-corrupted degradation analyses.
- `route_cards.json`: frozen route definitions and provenance.
- `clean_traces.jsonl` and `robustness_traces.jsonl`: row-level release traces.
- `HYPOTHESIS_RESULT.json`: confirmatory pass/fail outcome.
- `_SUCCESS.json`: experiment completion gate, not a positive-result claim.
"""

    def _dataset_readme(self, hypothesis_pass: bool) -> str:
        outcome = "passed" if hypothesis_pass else "did not pass"
        return f"""---
pretty_name: E2AM-MemRAG Traces
tags:
- rag
- memory
- green-ai
- energy-efficiency
- reproducibility
---

# E2AM-MemRAG Traces

{MANAGED_README_MARKER}

The frozen `{self.experiment_id}` experiment is complete and remotely verified.
The separately predeclared confirmatory hypothesis **{outcome}**; completion is
not presented as a positive result.

Human-readable paper artifacts are under
`experiments/{self.experiment_id}/paper/`. The complete 23-branch verified copy is
on branch `{self.destination_branch}`. Original worker branches and legacy
branches remain untouched.
"""

    def _finalize_destination(self, release: Mapping[str, Any]) -> dict[str, Any]:
        if set(self.progress["completed"]) != set(self.expected_branches):
            missing = sorted(set(self.expected_branches) - set(self.progress["completed"]))
            raise RuntimeError("Cannot finalize incomplete consolidation: " + ", ".join(missing))
        paper_files = self._paper_files_from_stage09()
        hypothesis = json.loads(paper_files["HYPOTHESIS_RESULT.json"].read_bytes())
        hypothesis_pass = bool(hypothesis.get("hypothesis_pass", False))
        manifest = {
            "schema_version": 1,
            "experiment_id": self.experiment_id,
            "source_lock_sha256": self.source_lock_sha256,
            "source_release_pointer": dict(release),
            "source_branches": self.progress["completed"],
            "source_branch_count": len(self.progress["completed"]),
            "source_artifact_records": sum(
                entry["artifact_records"] for entry in self.progress["completed"].values()
            ),
            "source_artifact_bytes": sum(
                entry["artifact_bytes"] for entry in self.progress["completed"].values()
            ),
            "excluded_legacy_branches": self.progress["excluded_legacy_branches"],
            "source_branches_modified": False,
            "hypothesis_pass": hypothesis_pass,
            "completion_is_independent_of_hypothesis": True,
        }
        manifest_bytes = canonical_json(manifest)
        success = {
            "schema_version": 1,
            "status": "COMPLETE",
            "experiment_id": self.experiment_id,
            "source_branch_count": manifest["source_branch_count"],
            "source_artifact_records": manifest["source_artifact_records"],
            "source_artifact_bytes": manifest["source_artifact_bytes"],
            "consolidation_manifest_sha256": sha256_bytes(manifest_bytes),
            "hypothesis_pass": hypothesis_pass,
            "completion_is_independent_of_hypothesis": True,
            "source_branches_modified": False,
        }
        unhashed = dict(success)
        success["gate_sha256"] = sha256_bytes(canonical_json(unhashed))
        success_bytes = canonical_json(success)
        manifest_local = self._local_file("CONSOLIDATION_MANIFEST.json", manifest_bytes)
        success_local = self._local_file("CONSOLIDATION_SUCCESS.json", success_bytes)
        branch_readme = self._local_file(
            "FINAL_BRANCH_README.md", self._branch_readme(status="COMPLETE").encode("utf-8")
        )
        paper_readme = self._local_file(
            "PAPER_RELEASE_README.md", self._paper_readme(hypothesis_pass).encode("utf-8")
        )
        candidate_progress = json.loads(canonical_json(self.progress))
        if candidate_progress["status"] != "COMPLETE":
            candidate_progress["status"] = "COMPLETE"
            candidate_progress["updated_at"] = utc_now()
            # The two reservations cover destination finalization and the following
            # main publication/verification. They are persisted for crash recovery;
            # the active session continues to account actual calls normally.
            candidate_progress["hub_budget_events"] = (
                self._budget_snapshot_with_reservations(42, 70)
            )
        progress_bytes = canonical_json(candidate_progress)
        progress_local = self._local_file("PROGRESS.json", progress_bytes)
        files: dict[str, Path] = {
            f"{self.remote_root}/CONSOLIDATION_MANIFEST.json": manifest_local,
            f"{self.remote_root}/_SUCCESS.json": success_local,
            f"{self.remote_root}/README.md": branch_readme,
            f"{self.remote_root}/paper/README.md": paper_readme,
            f"{self.remote_root}/PROGRESS.json": progress_local,
        }
        for logical, path in paper_files.items():
            files[f"{self.remote_root}/paper/{logical}"] = path
        self.destination_head = self._commit_verified(
            revision=self.destination_branch,
            parent_commit=self.destination_head,
            files=files,
            message=f"Finalize {self.experiment_id} verified consolidated release",
        )
        self.progress = candidate_progress
        self.last_verified_progress_bytes = progress_bytes
        return {
            "manifest": manifest,
            "manifest_bytes": manifest_bytes,
            "success": success,
            "success_bytes": success_bytes,
            "paper_files": paper_files,
            "paper_readme": paper_readme,
            "hypothesis_pass": hypothesis_pass,
        }

    def _publish_main(self, final: Mapping[str, Any]) -> str:
        main_head = self._repo_head("main")
        base = f"experiments/{self.experiment_id}"
        pointer = {
            "schema_version": 1,
            "experiment_id": self.experiment_id,
            "consolidation_branch": self.destination_branch,
            "consolidation_commit_sha": self.destination_head,
            "consolidation_root": self.remote_root,
            "consolidation_manifest_sha256": sha256_bytes(final["manifest_bytes"]),
            "consolidation_success_sha256": sha256_bytes(final["success_bytes"]),
            "paper_prefix": f"{base}/paper",
            "source_branch_count": final["manifest"]["source_branch_count"],
            "source_artifact_records": final["manifest"]["source_artifact_records"],
            "source_artifact_bytes": final["manifest"]["source_artifact_bytes"],
            "source_branches_modified": False,
        }
        pointer_local = self._local_file(
            "CONSOLIDATED_RELEASE.json", canonical_json(pointer)
        )
        manifest_local = self._local_file(
            "MAIN_CONSOLIDATION_MANIFEST.json", final["manifest_bytes"]
        )
        files: dict[str, Path] = {
            f"{base}/CONSOLIDATED_RELEASE.json": pointer_local,
            f"{base}/CONSOLIDATION_MANIFEST.json": manifest_local,
            f"{base}/PAPER_RELEASE_README.md": final["paper_readme"],
        }
        for logical, path in final["paper_files"].items():
            files[f"{base}/paper/{logical}"] = path
        if f"{base}/RELEASE.json" in files:
            raise RuntimeError("Frozen RELEASE.json must never be overwritten")

        existing_files = set(
            self._call(
                lambda: self.api.list_repo_files(
                    repo_id=self.repo_id,
                    repo_type=self.repo_type,
                    revision=main_head,
                ),
                weight=1,
                reason="inspect-main-before-publication",
            )
        )
        root_readme = self._local_file(
            "DATASET_README.md",
            self._dataset_readme(final["hypothesis_pass"]).encode("utf-8"),
        )
        if "README.md" not in existing_files:
            files["README.md"] = root_readme

        changed: dict[str, Path] = {}
        for remote_path, local_path in files.items():
            if remote_path not in existing_files:
                changed[remote_path] = local_path
                continue
            remote = self._download(filename=remote_path, revision=main_head)
            if remote != local_path.read_bytes():
                raise RuntimeError(
                    "MAIN_PUBLICATION_CONFLICT: existing file differs and will not be overwritten: "
                    + remote_path
                )
        if changed:
            main_head = self._commit_verified(
                revision="main",
                parent_commit=main_head,
                files=changed,
                message=f"Publish visible {self.experiment_id} paper and consolidation release",
            )
        for remote_path, local_path in files.items():
            remote = self._download(filename=remote_path, revision=main_head)
            if remote != local_path.read_bytes():
                raise RuntimeError(f"Main publication verification failed: {remote_path}")
        return main_head

    def _safe_stop(self) -> bool:
        if (
            self.destination_head is None
            or self.last_verified_progress_bytes is None
        ):
            return True
        remote = self._download(
            filename=f"{self.remote_root}/PROGRESS.json",
            revision=self.destination_head,
        )
        return remote == self.last_verified_progress_bytes

    def run(self) -> dict[str, Any]:
        self._load_hub()
        branches, legacy = self._list_refs()
        release = self._verify_release_pointer()
        self._initialize_destination(branches=branches, legacy=legacy)
        try:
            for lock in self.source_lock:
                self._process_branch(lock)
            final = self._finalize_destination(release)
            main_commit = self._publish_main(final)
        except KeyboardInterrupt:
            verified = self._safe_stop()
            print("SAFE_STOP_VERIFIED" if verified else "SAFE_STOP_FAILED", flush=True)
            raise
        except BaseException:
            try:
                verified = self._safe_stop()
                print("SAFE_STOP_VERIFIED" if verified else "SAFE_STOP_FAILED", flush=True)
            except Exception as stop_error:
                print(
                    "SAFE_STOP_FAILED",
                    {"reason": type(stop_error).__name__},
                    flush=True,
                )
            raise
        report = {
            "go": True,
            "experiment_id": self.experiment_id,
            "source_branch_count": final["manifest"]["source_branch_count"],
            "source_artifact_records": final["manifest"]["source_artifact_records"],
            "source_artifact_bytes": final["manifest"]["source_artifact_bytes"],
            "excluded_legacy_branches": final["manifest"]["excluded_legacy_branches"],
            "hypothesis_pass": final["hypothesis_pass"],
            "completion_is_independent_of_hypothesis": True,
            "consolidation_branch": self.destination_branch,
            "consolidation_commit_sha": self.destination_head,
            "main_commit_sha": main_commit,
            "source_branches_modified": False,
            "remote_verified": True,
            "main_visible": True,
        }
        print("CONSOLIDATION_COMPLETE", report, flush=True)
        return report


def run_consolidation(config: Mapping[str, Any], *, hf_token: str) -> dict[str, Any]:
    """Run or resume the exact locked v3r1 consolidation."""

    return Consolidator(config, hf_token=hf_token).run()


In [ ]:
# Fail before any Hub mutation if local capacity or the frozen lock is inconsistent.
import shutil

disk = shutil.disk_usage("/kaggle/working")
assert len(SOURCE_RELEASE_LOCK) == 23
assert len({item["branch"] for item in SOURCE_RELEASE_LOCK}) == 23
assert sum(item["artifact_records"] for item in SOURCE_RELEASE_LOCK) == 282
assert sum(item["artifact_bytes"] for item in SOURCE_RELEASE_LOCK) == 127_554_473
assert CONFIG["hub_capacity"] <= 96  # preserves at least 25% of 128/hour
if disk.free < 1_000_000_000:
    raise RuntimeError(
        "STORAGE_SAFE_STOP: less than 1 GB is free under /kaggle/working. "
        "Start a fresh CPU session; no remote branch was changed."
    )
print("CONSOLIDATION_PREFLIGHT_GO", {
    "free_gib": round(disk.free / (1024 ** 3), 2),
    "source_branches": 23,
    "expected_mib": round(127_554_473 / (1024 ** 2), 3),
    "gpu_required": False,
})


In [ ]:
# Run or resume. A second writer, mixed release, checksum mismatch, or
# conflicting main file causes a safe stop instead of an overwrite.
FINAL_REPORT = run_consolidation(CONFIG, hf_token=HF_TOKEN)


In [ ]:
# Final publication gate.
assert FINAL_REPORT["go"] is True
assert FINAL_REPORT["remote_verified"] is True
assert FINAL_REPORT["main_visible"] is True
assert FINAL_REPORT["source_branch_count"] == 23
assert FINAL_REPORT["source_artifact_records"] == 282
assert FINAL_REPORT["source_artifact_bytes"] == 127_554_473
assert FINAL_REPORT["source_branches_modified"] is False
assert FINAL_REPORT["completion_is_independent_of_hypothesis"] is True

print("FINAL_GO", {
    "consolidation_branch": FINAL_REPORT["consolidation_branch"],
    "consolidation_commit": FINAL_REPORT["consolidation_commit_sha"],
    "main_commit": FINAL_REPORT["main_commit_sha"],
    "paper_url": "https://huggingface.co/datasets/Shanmuk4622/E2AM-MemRAG-Traces/tree/main/experiments/e2am-memrag-v3r1/paper",
    "hypothesis_pass": FINAL_REPORT["hypothesis_pass"],
    "all_source_branches_preserved": True,
})
